# WP05B — One small Tripoli export pilot / آزمایش خروجی یک قطعه

این نوت‌بوک پشتیبان WP05 را می‌گیرد و **فقط یک قطعهٔ ۱۰۲۴×۱۰۲۴ پیکسل** را از همان نقشهٔ انتخاب‌شده صادر می‌کند. کد GEE، روش مدل و شبکهٔ ۱۰ متری تغییر نمی‌کنند. درخواست بزرگ قبلی فقط خوانده می‌شود؛ لغو یا دوباره ارسال نمی‌شود.

**شروع تازه:** سلول‌های ۱ تا ۷ را به‌ترتیب اجرا کن. ZIP پشتیبان دیشب (`wp05-export-...zip`) را در سلول ۳ بارگذاری کن. سلول ۵ یک درخواست کوچک جدید می‌سازد.

**پس از قطع Colab:** سلول‌های ۱ تا ۴ را اجرا و ZIP جدید `wp05-pilot-...zip` را بارگذاری کن؛ سپس به سلول ۶ برو. برای صرفاً خواندن وضعیت، سلول ۵ لازم نیست.

The default tile is centred on the AOI's projected centroid and checked for positive AOI overlap. It is a delivery smoke test, not an accuracy sample or a guarantee of fast computation. Retained graphs can still recompute the selected classifier. No bulk tile scheduler is enabled. Live completion and TIFF inspection remain pending until you run this notebook.


## 1. نصب نسخهٔ ثابت / Install in a fresh runtime

In [ ]:
import sys, subprocess
PILOT_COMMIT = "1dbbb180c32d4b23f91643439f90126fadd6dc5c"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
    "git+https://github.com/hamedsabzchi/libya-automated-land-cover-mapping.git@" + PILOT_COMMIT])
import ee, libya_landcover
if libya_landcover.__version__ != "0.5.1":
    raise RuntimeError("Use a fresh runtime: an older package is still imported")
assert ee.__version__ == "1.7.48"
print("Pilot service:", libya_landcover.__version__, PILOT_COMMIT)


## 2. ورود به همان حساب Google / Authenticate and initialize

In [ ]:
PROJECT = "practical-proxy-441422-n6"
ee.Authenticate()
ee.Initialize(project=PROJECT)
print("Connected:", PROJECT)


## 3. بارگذاری یک ZIP / Upload one backup
Accepts your completed WP04 ZIP, a WP05 export backup, or a new pilot handoff. The importer reads only checked mapping graphs/JSON; it does not execute uploaded Python, JavaScript or SQLite. For recovery, prefer the newest pilot ZIP so the exact window, operation and history are retained.


In [ ]:
import json
from pathlib import Path
from google.colab import files
from libya_landcover.bundle import write_json
from libya_landcover.job_store import JobStore
from libya_landcover.jobs import checked_mapping_bundle
from libya_landcover.exports import submit_export, poll_export
from libya_landcover.pilot import (adopt_archive, plan_pilot, create_pilot,
    recover_pilot, save_diagnostic, save_handoff)

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one ZIP")
source_bytes = next(iter(uploaded.values()))
source = Path("/content/pilot-source.zip")
source.write_bytes(source_bytes)
del uploaded, source_bytes
store = JobStore("/content/tripoli-pilot-store")
imported = adopt_archive(store, source)
mapping = imported["mapping"]
checked = checked_mapping_bundle(store, mapping)
result = checked["result"].to_dict()
assert checked["manifest"]["project"] == PROJECT
assert result["request"]["aoi_asset"] == "projects/practical-proxy-441422-n6/assets/Libya/Tripoli"
assert result["request"]["training_asset"] == "projects/practical-proxy-441422-n6/assets/Libya/TD3-attept1-tripoli"
finals = [item for item in result["outputs"] if item["kind"] == "final_map"]
if len(finals) != 1:
    raise ValueError("Expected one retained final map")
artifact = finals[0]
print("Mapping imported:", mapping["state"], "source run:", result["run_id"])
print("Pilot recovery:", imported["context"] is not None)


## 4. مشاهدهٔ درخواست قبلی و آماده‌کردن قطعه / Read old task and prepare or recover
This cell performs only geometry/status reads and local preparation. The window is at most 1,048,576 pixels (10.24 km on each side at 10 m); it retains the parent pixel lattice. The overlap check does not prove valid class pixels exist. Outside-AOI pixels are NoData=0. If the default centroid tile has no AOI overlap, use an explicit `[longitude, latitude]` inside your region.

On recovery, the saved plan takes precedence over the selection parameters below; the known remote operation is verified and reattached. No new export is submitted. A full-area task completing in the meantime is reported independently.


In [ ]:
ORIGINAL_OPERATION = (
    "projects/practical-proxy-441422-n6/operations/"
    "WF6TW2SC56MZK76YWEP4T42V"
)
try:
    original_status = ee.data.getOperation(ORIGINAL_OPERATION)
    print("Original full-area task (read only):", json.dumps(original_status, indent=2))
except Exception as exc:
    original_status = {"name": ORIGINAL_OPERATION, "lookup_error_type": type(exc).__name__}
    print("Original status lookup unavailable:", type(exc).__name__)

CENTER_WGS84 = None  # Default: AOI centroid. Optional explicit [longitude, latitude].
SIDE_PIXELS = 1024  # Maximum; no automatic increase or bulk task creation.
if imported["context"] is not None:
    plan = imported["context"]["plan"]
    pilot = recover_pilot(store, mapping["job_id"], imported["context"])
else:
    plan = plan_pilot(store, mapping["job_id"], side=SIDE_PIXELS, center_wgs84=CENTER_WGS84)
    pilot = create_pilot(store, mapping["job_id"], artifact["artifact_id"], plan)
PILOT_JOB_ID = pilot["job_id"]
folder = store.folder(PILOT_JOB_ID)
folder.mkdir(parents=True, exist_ok=True)
write_json(folder / "original-operation.json", original_status)
if imported["previous_manifest"] is not None:
    write_json(folder / "source-manifest.json", imported["previous_manifest"])

def checkpoint():
    latest = store.get(PILOT_JOB_ID)
    path = Path("/content/wp05-pilot-" + PILOT_JOB_ID + "-" + str(latest["revision"]) + ".zip")
    save_handoff(store, PILOT_JOB_ID, plan, path, service_commit=PILOT_COMMIT,
                 original_operation=ORIGINAL_OPERATION)
    return path

checkpoint()
print("Pilot plan:", json.dumps(plan, indent=2))
print("Pilot job:", json.dumps(pilot, indent=2))
print("Preparation/recovery only. Cell 5 explicitly submits; cell 6 reads status.")


## 5. ارسال تنها یک قطعه / Explicitly submit the pilot
Run for the initial pilot. After restoring a known operation, skip to cell 6. Repeating this cell uses the same job/request identity; it never creates an automatic new attempt. A failed/cancelled pilot is retained for diagnosis. The original full-area task is not cancelled or resubmitted.

Save a backup with cell 7 immediately after submission, then again after status changes. No performance or completion time is promised.


In [ ]:
if "PILOT_JOB_ID" not in globals():
    raise RuntimeError("Run cells 1–4 first. After a reset, upload the pilot backup in cell 3.")
try:
    pilot = submit_export(store, PILOT_JOB_ID)
    print(json.dumps(pilot, indent=2))
finally:
    checkpoint()


## 6. وضعیت و جزئیات واقعی / Poll and record progress, attempts and errors
Repeat this cell occasionally, not the submit cell. Percentages are service estimates, not time estimates. A higher attempt number can explain a progress reset; the retry cause may not be reported. If the pilot also runs slowly, preserve the evidence before deciding on a different delivery strategy; do not automatically start more tiles.


In [ ]:
if "PILOT_JOB_ID" not in globals():
    raise RuntimeError("Session was reset or not prepared. Run cells 1–4 with the pilot ZIP, then this cell. Do not Run all.")
try:
    pilot = poll_export(store, PILOT_JOB_ID)
except Exception as exc:
    print("Status lookup failed; remote state is unknown:", type(exc).__name__)
    pilot = store.get(PILOT_JOB_ID)
diagnostic = save_diagnostic(store, PILOT_JOB_ID)
checkpoint()
print("Recorded job:", json.dumps({key: pilot.get(key) for key in (
    "job_id", "state", "operation_name", "remote_metadata", "error", "destination_uris")}, indent=2))
print("Latest raw server observation:", json.dumps(diagnostic, indent=2, ensure_ascii=False))
if pilot["state"] == "completed":
    print("Download the pilot TIFF from these service-provided destinations:")
    for uri in pilot["destination_uris"]:
        print(uri)
    print("If there is no link, find this exact folder in Drive:",
          pilot.get("drive_folder", pilot["inputs"]["spec"]["folder_prefix"] + "_" + pilot["ee_request_id"]))
    print("Send the TIFF and the cell 7 ZIP for grid/encoding/runtime inspection.")
elif pilot["state"] in {"failed", "cancelled"}:
    print("Terminal state. Download cell 7 ZIP; do not start another pilot automatically.")
else:
    print("Not completed. Download a backup with cell 7 now, then poll again later.")


## 7. دریافت پشتیبان / Download the portable handoff
Run after submitting, before closing Colab, and after the terminal result. This is a scoped private ZIP with the selected mapping graphs, exact pilot plan, events and raw diagnostic records, including server errors. It excludes credentials and TIFF pixels. Keep the original WP05 backup too. On a new runtime, upload this pilot ZIP in cell 3 and execute cells 1–4, then cell 6.


In [ ]:
if "PILOT_JOB_ID" not in globals():
    raise RuntimeError("Run cells 1–4 with your backup to recover the pilot first.")
path = checkpoint()
files.download(str(path))
print("Backup saved:", path.name, "state:", store.get(PILOT_JOB_ID)["state"])


## Live acceptance and next decision

- Obtain a terminal pilot result and the actual TIFF. Confirm EPSG:32633, 10 m pixel alignment, expected bounded extent, one UInt8 class band and NoData=0. Confirm there are valid class pixels; an all-NoData tile is inconclusive.
- Compare available start/end/attempt metadata and elapsed time. EECU usage is a service compute metric, not a wall-clock duration or a monetary bill.
- Only after inspecting this pilot should we decide whether to design a complete grid and independently resumable exports. Smaller regions may still repeat expensive upstream training/aggregation computations. This notebook schedules no full grid.
- The unchanged GEE reference and Python model remain authoritative. This work does not revise the model's accuracy, seeds, training scope or temporal rules.
